# Task 3 · Cleaning Data — Café Sales

**Objective:** take the deliberately messy `dirty_cafe_sales.csv` (10,000 transactions) and turn it into a clean, analysis-ready dataset, documenting every decision.

**Stack:** Python · pandas · numpy · Jupyter

**Workflow**
1. Load the raw data and build a *data quality report*
2. Standardise formatting (whitespace, casing, placeholder tokens)
3. Handle missing data (column-by-column strategy)
4. Remove duplicates
5. Correct data types
6. Detect and treat outliers
7. Validate business rules (`Total = Quantity × Price`)
8. Before vs. after summary
9. Save the cleaned CSV

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

RAW_PATH = "dirty_cafe_sales.csv"
CLEAN_PATH = "clean_cafe_sales.csv"

## 1 · Load the raw data

Everything is loaded as **string** with `keep_default_na=False`. This is deliberate: the file hides missing values in three different ways (empty cells, the text `UNKNOWN`, and the text `ERROR`), and letting pandas auto-parse would silently coerce some of them and hide the problem. We keep the raw text so we can measure the mess honestly.

In [2]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
df_before = raw.copy()          # untouched copy for the before/after comparison
print(raw.shape)
raw.head(10)

(10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,,2023-03-31
6,TXN_4433211,UNKNOWN,3,3.0,9.0,ERROR,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,UNKNOWN,2023-10-28
8,TXN_4717867,,5,3.0,15.0,,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5,4.0,20.0,,In-store,2023-12-31


## 2 · Data quality report (before cleaning)

We look at five things: missing values (in every disguise), duplicates, data type problems, inconsistent categories, and value-range anomalies.

In [3]:
PLACEHOLDERS = ["", "UNKNOWN", "ERROR"]

# --- 2a. Missing values in every disguise -------------------------------------------------
missing_report = pd.DataFrame({
    "empty ('')":      (raw == "").sum(),
    "'UNKNOWN'":       (raw == "UNKNOWN").sum(),
    "'ERROR'":         (raw == "ERROR").sum(),
})
missing_report["total missing"] = missing_report.sum(axis=1)
missing_report["% missing"] = (missing_report["total missing"] / len(raw) * 100).round(2)
missing_report

,empty (''),'UNKNOWN','ERROR',total missing,% missing
Transaction ID,0,0,0,0,0.00
Item,333,344,292,969,9.69
Quantity,138,171,170,479,4.79
Price Per Unit,179,164,190,533,5.33
Total Spent,173,165,164,502,5.02
Payment Method,2579,293,306,3178,31.78
Location,3265,338,358,3961,39.61
Transaction Date,159,159,142,460,4.60


In [4]:
# --- 2b. Duplicates ---------------------------------------------------------------------------
print("Exact duplicate rows            :", raw.duplicated().sum())
print("Duplicate Transaction IDs       :", raw["Transaction ID"].duplicated().sum())
print("Duplicates ignoring Transaction ID:", raw.drop(columns="Transaction ID").duplicated().sum(),
      "(expected to be non-zero by chance: ~10k rows, few distinct values)")

# --- 2c. Data types ----------------------------------------------------------------------------
print()
print("Current dtypes:")
print(raw.dtypes.to_string())

Exact duplicate rows            : 0
Duplicate Transaction IDs       : 0
Duplicates ignoring Transaction ID: 141 (expected to be non-zero by chance: ~10k rows, few distinct values)

Current dtypes:
Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str


In [5]:
# --- 2d. Distinct values in the categorical columns (spot inconsistent labels) ---------
for col in ["Item", "Payment Method", "Location"]:
    print(f"\n{col}:")
    print(raw[col].value_counts(dropna=False).to_string())


Item:
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
             333
ERROR        292

Payment Method:
Payment Method
                  2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293

Location:
Location
            3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338


In [6]:
# --- 2e. Value ranges on numeric columns (ignoring placeholders for now) ---------------
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]
tmp = raw[numeric_cols].apply(pd.to_numeric, errors="coerce")
print(tmp.describe().T)

# Date range
dates = pd.to_datetime(raw["Transaction Date"], errors="coerce", format="%Y-%m-%d")
print("\nDate range:", dates.min().date(), "→", dates.max().date())
print("Unparseable / missing dates:", dates.isna().sum())

# Item ↔ price relationship (is each item sold at one fixed price?)
price_by_item = (raw.assign(p=pd.to_numeric(raw["Price Per Unit"], errors="coerce"))
                    .query("Item not in @PLACEHOLDERS")
                    .groupby("Item")["p"].agg(["nunique", "min", "max"]))
print("\nPrice per item (nunique=1 means a fixed menu price):")
print(price_by_item)

                 count      mean       std  min  25%  50%   75%   max
Quantity        9521.0  3.028463  1.419007  1.0  2.0  3.0   4.0   5.0
Price Per Unit  9467.0  2.949984  1.278450  1.0  2.0  3.0   4.0   5.0
Total Spent     9498.0  8.924352  6.009919  1.0  4.0  8.0  12.0  25.0

Date range: 2023-01-01 → 2023-12-31
Unparseable / missing dates: 460

Price per item (nunique=1 means a fixed menu price):
          nunique  min  max
Item                       
Cake            1  3.0  3.0
Coffee          1  2.0  2.0
Cookie          1  1.0  1.0
Juice           1  3.0  3.0
Salad           1  5.0  5.0
Sandwich        1  4.0  4.0
Smoothie        1  4.0  4.0
Tea             1  1.5  1.5


### Findings

- **No true `NaN`s** — missingness is encoded as empty strings, `UNKNOWN` and `ERROR`. All three mean "value not available" and will be treated identically.
- **Payment Method** and **Location** are the worst affected (roughly a quarter to a third of rows).
- **Every column is `object`/string**, so numbers and dates cannot be used until converted.
- **No duplicate rows and no duplicate Transaction IDs** in the raw file.
- Each menu **item has exactly one price**, and `Total Spent = Quantity × Price Per Unit`. This is the key insight: Item, Quantity, Price and Total are mutually dependent, so many missing values can be **recovered exactly** rather than guessed.

## 3 · Standardisation

- Strip whitespace from all text.
- Normalise casing of categorical labels (`Title Case`) so that variants such as `cash` / `CASH` / `Cash` cannot create phantom categories.
- Convert the three placeholder tokens (`''`, `UNKNOWN`, `ERROR`) to a real missing value (`NaN`) so pandas can reason about them.
- Dates: parse to `datetime64`; anything unparseable becomes `NaT`.

In [7]:
df = raw.copy()

# strip whitespace everywhere
df = df.apply(lambda s: s.str.strip())

# unify placeholders (case-insensitive) → NaN
df = df.replace(r"(?i)^(unknown|error|nan|none|null|n/a)?$", np.nan, regex=True)

# categorical labels → consistent Title Case
for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].str.title()

# numeric columns
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# dates
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce", format="%Y-%m-%d")

print("Distinct labels after standardisation:")
for col in ["Item", "Payment Method", "Location"]:
    print(f"  {col}: {sorted(df[col].dropna().unique())}")
print()
print(df.isna().sum().to_string())

Distinct labels after standardisation:
  Item: ['Cake', 'Coffee', 'Cookie', 'Juice', 'Salad', 'Sandwich', 'Smoothie', 'Tea']
  Payment Method: ['Cash', 'Credit Card', 'Digital Wallet']
  Location: ['In-Store', 'Takeaway']



Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460


## 4 · Missing data handling

Dropping every row with any gap would throw away most of the data (see the rates above), and blanket mean/mode imputation would invent values. So each column gets its own strategy:

| Column | Strategy | Justification |
|---|---|---|
| **Price Per Unit** | Recover from the item's fixed menu price; else `Total ÷ Quantity` | Each item has exactly one price, so this is an exact reconstruction, not an estimate |
| **Item** | Recover only when the price identifies a single item; otherwise label `Unknown` | Prices 1.0, 1.5, 2.0 and 5.0 are unique to one item. Prices 3.0 (Cake/Juice) and 4.0 (Sandwich/Smoothie) are ambiguous, so guessing (e.g. mode imputation) would fabricate sales for a specific product |
| **Quantity** | `Total ÷ Price` | Exact arithmetic identity |
| **Total Spent** | `Quantity × Price` | Exact arithmetic identity |
| **Payment Method**, **Location** | Label as `Unknown` (keep the row) | 25–33% missing — far too many to delete, and mode imputation would add thousands of fake "Digital Wallet" / "Takeaway" transactions and distort any analysis. `Unknown` preserves the row's numeric information and is transparent |
| **Transaction Date** | Leave as `NaT` | Rows are not in time order, so forward-fill would assign wrong dates; there is no other column to infer from. Missing dates are only ~4.6% of rows, so leaving them blank is the honest choice (they simply drop out of time-series analysis) |
| **Rows where Total Spent is still unrecoverable** | Delete | If two of {Quantity, Price, Total} are missing and Item cannot supply the price, the transaction value is unknowable; a row with no revenue figure is of no use for sales analysis |

In [8]:
before_fill = df.isna().sum()

# 4a. menu price per item (learned from rows where both are known; verified fixed above)
menu = (df.dropna(subset=["Item", "Price Per Unit"])
          .groupby("Item")["Price Per Unit"].agg(lambda s: s.mode().iloc[0]))
assert (df.dropna(subset=["Item", "Price Per Unit"]).groupby("Item")["Price Per Unit"].nunique() == 1).all(), \
    "An item has more than one price - menu lookup would be unsafe"
print("Menu prices learned from the data:")
print(menu.to_string())

# prices that identify exactly one item
price_to_items = menu.reset_index().groupby("Price Per Unit")["Item"].agg(list)
unique_price_to_item = {p: items[0] for p, items in price_to_items.items() if len(items) == 1}
print("\nPrices that map to a single item:", unique_price_to_item)
print("Ambiguous prices:", {p: i for p, i in price_to_items.items() if len(i) > 1})

Menu prices learned from the data:
Item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5

Prices that map to a single item: {1.0: 'Cookie', 1.5: 'Tea', 2.0: 'Coffee', 5.0: 'Salad'}
Ambiguous prices: {3.0: ['Cake', 'Juice'], 4.0: ['Sandwich', 'Smoothie']}


In [9]:
def recover(d):
    d = d.copy()
    # price from item
    m = d["Price Per Unit"].isna() & d["Item"].notna()
    d.loc[m, "Price Per Unit"] = d.loc[m, "Item"].map(menu)
    # price from total / quantity
    m = d["Price Per Unit"].isna() & d["Total Spent"].notna() & d["Quantity"].notna()
    d.loc[m, "Price Per Unit"] = d.loc[m, "Total Spent"] / d.loc[m, "Quantity"]
    # quantity from total / price
    m = d["Quantity"].isna() & d["Total Spent"].notna() & d["Price Per Unit"].notna()
    d.loc[m, "Quantity"] = d.loc[m, "Total Spent"] / d.loc[m, "Price Per Unit"]
    # total from quantity * price
    m = d["Total Spent"].isna() & d["Quantity"].notna() & d["Price Per Unit"].notna()
    d.loc[m, "Total Spent"] = d.loc[m, "Quantity"] * d.loc[m, "Price Per Unit"]
    # item from unambiguous price
    m = d["Item"].isna() & d["Price Per Unit"].isin(unique_price_to_item.keys())
    d.loc[m, "Item"] = d.loc[m, "Price Per Unit"].map(unique_price_to_item)
    return d

# run until nothing more can be recovered (one pass can unlock the next)
for i in range(5):
    nxt = recover(df)
    changed = not nxt.equals(df)
    df = nxt
    if not changed:
        break
print(f"Recovery converged after {i+1} pass(es)")

recovered = before_fill - df.isna().sum()
print("\nValues recovered by arithmetic / menu lookup:")
print(recovered.to_string())

Recovery converged after 2 pass(es)

Values recovered by arithmetic / menu lookup:
Transaction ID        0
Item                489
Quantity            456
Price Per Unit      527
Total Spent         479
Payment Method        0
Location              0
Transaction Date      0


In [10]:
# 4b. rows whose transaction value is still unknowable → delete
unrecoverable = df["Total Spent"].isna()
rows_dropped_missing = int(unrecoverable.sum())
df = df.loc[~unrecoverable].copy()
print("Rows deleted because Total Spent could not be recovered:", rows_dropped_missing)

# 4c. remaining numeric gaps are harmless once Total is known, but check
print("\nRemaining nulls in numeric columns:")
print(df[["Quantity", "Price Per Unit", "Total Spent"]].isna().sum().to_string())

Rows deleted because Total Spent could not be recovered: 23

Remaining nulls in numeric columns:
Quantity          3
Price Per Unit    3
Total Spent       0


In [11]:
# 4d. categorical gaps → explicit 'Unknown' label
for col in ["Item", "Payment Method", "Location"]:
    n = df[col].isna().sum()
    df[col] = df[col].fillna("Unknown")
    print(f"{col}: {n} labelled 'Unknown'")

print("\nTransaction Date left as NaT:", df["Transaction Date"].isna().sum())

Item: 477 labelled 'Unknown'
Payment Method: 3169 labelled 'Unknown'
Location: 3953 labelled 'Unknown'

Transaction Date left as NaT: 460


**Note on the remaining numeric nulls:** if any `Quantity` or `Price Per Unit` gaps remain, they are rows where `Total Spent` is known but the split into quantity × price cannot be determined. The revenue figure is still valid, so we keep the row rather than invent a value. The count is shown above.

## 5 · Duplicate removal

Checked on the full row and on `Transaction ID` (the natural key). Duplicates are checked *after* standardisation, because inconsistent formatting can hide duplicates (e.g. `cash` vs `Cash`).

In [12]:
dups_full = int(df.duplicated().sum())
dups_id = int(df["Transaction ID"].duplicated().sum())
print("Exact duplicate rows after standardisation :", dups_full)
print("Duplicate Transaction IDs                  :", dups_id)

rows_before_dedup = len(df)
df = df.drop_duplicates().drop_duplicates(subset="Transaction ID", keep="first").reset_index(drop=True)
duplicates_removed = rows_before_dedup - len(df)
print("Duplicate rows removed                     :", duplicates_removed)

Exact duplicate rows after standardisation : 0
Duplicate Transaction IDs                  : 0
Duplicate rows removed                     : 0


The raw file contains **no duplicate transactions**, so zero rows are removed. We still run the check (and the removal code) so the pipeline stays correct if the data is refreshed.

## 6 · Data type correction

| Column | Target dtype | Why |
|---|---|---|
| Transaction ID | `string` | Identifier — never used in arithmetic |
| Item, Payment Method, Location | `category` | Small fixed label set; saves memory, enables ordering/grouping |
| Quantity | `Int64` (nullable int) | Whole units; nullable because a few values may remain unknown |
| Price Per Unit, Total Spent | `float64` | Monetary values |
| Transaction Date | `datetime64[ns]` | Enables time-series operations |

In [13]:
df["Transaction ID"] = df["Transaction ID"].astype("string")
for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].astype("category")

# Quantity must be whole numbers - verify before casting
q = df["Quantity"].dropna()
assert (q == q.round()).all(), "Non-integer quantity found"
df["Quantity"] = df["Quantity"].round().astype("Int64")

df["Price Per Unit"] = df["Price Per Unit"].astype("float64")
df["Total Spent"] = df["Total Spent"].astype("float64")

df.dtypes

Transaction ID              string
Item                      category
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[us]
dtype: object

## 7 · Outlier detection

Both methods are applied to every numeric column:
- **IQR rule:** outlier if value < Q1 − 1.5·IQR or > Q3 + 1.5·IQR
- **Z-score:** outlier if |z| > 3

In [14]:
def outlier_report(frame, cols):
    rows = []
    for c in cols:
        s = frame[c].dropna().astype(float)
        q1, q3 = s.quantile([.25, .75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        z = (s - s.mean()) / s.std()
        rows.append({
            "column": c, "min": s.min(), "max": s.max(), "mean": round(s.mean(), 3),
            "Q1": q1, "Q3": q3, "IQR lower": lo, "IQR upper": hi,
            "IQR outliers": int(((s < lo) | (s > hi)).sum()),
            "Z>3 outliers": int((z.abs() > 3).sum()),
        })
    return pd.DataFrame(rows).set_index("column")

outlier_report(df, ["Quantity", "Price Per Unit", "Total Spent"])

,min,max,mean,Q1,Q3,IQR lower,IQR upper,IQR outliers,Z>3 outliers
column,,,,,,,,,
Quantity,1.0,5.0,3.025,2.0,4.0,-1.0,7.0,0,0
Price Per Unit,1.0,5.0,2.947,2.0,4.0,-1.0,7.0,0,0
Total Spent,1.0,25.0,8.930,4.0,12.0,-8.0,24.0,269,0


In [15]:
# Inspect the flagged Total Spent values rather than trusting the statistic blindly
s = df["Total Spent"]
q1, q3 = s.quantile([.25, .75]); hi = q3 + 1.5 * (q3 - q1)
flagged = df[s > hi]
print(f"Total Spent above IQR upper fence ({hi:.2f}): {len(flagged)} rows")
print(flagged.groupby(["Item", "Quantity", "Price Per Unit", "Total Spent"], observed=True).size()
             .rename("rows").reset_index().to_string(index=False))

Total Spent above IQR upper fence (24.00): 269 rows
 Item  Quantity  Price Per Unit  Total Spent  rows
Salad         5             5.0         25.0   268


### Outlier decision: **retain all values**

| Column | Decision | Reason |
|---|---|---|
| **Quantity** | Retain | Range is a discrete 1–5 with no values outside it; nothing is flagged |
| **Price Per Unit** | Retain | Values are fixed menu prices (1.0 – 5.0); they cannot be "extreme" |
| **Total Spent** | Retain | Any value flagged by IQR is a legitimate combination (e.g. 5 × 5.0 = 25.0 for a Salad). It is the mathematical maximum of the menu, not an error. Capping would *corrupt* correct revenue figures, and removing would bias sales downward |

Statistical outliers are not errors by default; here the flagged points are explained by the business rules, so no capping or removal is applied. (If a value *had* broken the rules, e.g. a quantity of 500, it would be removed or corrected.)

## 8 · Business-rule validation

Final integrity checks: `Total Spent = Quantity × Price Per Unit`, plausible ranges, and the date window.

In [16]:
calc = df["Quantity"].astype("float") * df["Price Per Unit"]
check = df.assign(calc=calc).dropna(subset=["calc"])
mismatch = (check["calc"] - check["Total Spent"]).abs() > 1e-9
print("Rows where Total != Quantity x Price:", int(mismatch.sum()), f"(of {len(check)} checkable rows)")
print("Quantity range      :", df["Quantity"].min(), "-", df["Quantity"].max())
print("Price range         :", df["Price Per Unit"].min(), "-", df["Price Per Unit"].max())
print("Negative totals     :", int((df["Total Spent"] < 0).sum()))
print("Date range          :", df["Transaction Date"].min().date(), "->", df["Transaction Date"].max().date())
print("Future-dated rows   :", int((df["Transaction Date"] > pd.Timestamp.today()).sum()))

Rows where Total != Quantity x Price: 0 (of 9974 checkable rows)
Quantity range      : 1 - 5
Price range         : 1.0 - 5.0
Negative totals     : 0
Date range          : 2023-01-01 -> 2023-12-31
Future-dated rows   : 0


## 9 · Before vs. after summary

*Dtype accuracy* = share of columns whose dtype matches the target schema from section 6. *Null count* includes the placeholder tokens (`''`, `UNKNOWN`, `ERROR`) in the **before** column, since those are nulls in disguise. In the **after** column, the explicit `Unknown` label in the categorical columns is a deliberate category, not a null, so only genuine gaps (`NaN`/`NaT`) are counted.

In [17]:
from pandas.api import types as pdt

# A column is "correct" if its dtype family matches the target schema.
# Text-like targets (ID, and categorical labels) accept string/object/category dtypes;
# numeric and datetime targets must be truly numeric / datetime.
CHECKS = {
    "Transaction ID":   pdt.is_string_dtype,
    "Item":             lambda s: pdt.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype),
    "Quantity":         pdt.is_integer_dtype,
    "Price Per Unit":   pdt.is_float_dtype,
    "Total Spent":      pdt.is_float_dtype,
    "Payment Method":   lambda s: pdt.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype),
    "Location":         lambda s: pdt.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype),
    "Transaction Date": pdt.is_datetime64_any_dtype,
}

def dtype_accuracy(frame):
    ok = sum(bool(fn(frame[c])) for c, fn in CHECKS.items())
    return ok, len(CHECKS)

# ----- BEFORE -----
null_before = int(df_before.isin(PLACEHOLDERS).sum().sum())
nulls_per_col_before = df_before.isin(PLACEHOLDERS).sum()
ok_b, n = dtype_accuracy(df_before)   # text columns are fine as text; the 4 numeric/date columns are not

# ----- AFTER -----
null_after = int(df.isna().sum().sum())
nulls_per_col_after = df.isna().sum()
ok_a, n = dtype_accuracy(df)

summary = pd.DataFrame({
    "Before": [len(df_before), null_before, int(df_before.duplicated().sum()),
               int(df_before["Transaction ID"].duplicated().sum()), f"{ok_b}/{n} ({ok_b/n:.0%})"],
    "After":  [len(df), null_after, int(df.duplicated().sum()),
               int(df["Transaction ID"].duplicated().sum()), f"{ok_a}/{n} ({ok_a/n:.0%})"],
}, index=["Row count", "Total null cells", "Duplicate rows", "Duplicate Transaction IDs", "Dtype accuracy"])
summary

,Before,After
Row count,10000,9977
Total null cells,10082,466
Duplicate rows,0,0
Duplicate Transaction IDs,0,0
Dtype accuracy,4/8 (50%),8/8 (100%)


In [18]:
per_col = pd.DataFrame({
    "Nulls before": nulls_per_col_before,
    "Nulls after":  nulls_per_col_after,
    "Dtype before": df_before.dtypes.astype(str),
    "Dtype after":  df.dtypes.astype(str),
})
per_col

,Nulls before,Nulls after,Dtype before,Dtype after
Transaction ID,0,0,str,string
Item,969,0,str,category
Quantity,479,3,str,Int64
Price Per Unit,533,3,str,float64
Total Spent,502,0,str,float64
Payment Method,3178,0,str,category
Location,3961,0,str,category
Transaction Date,460,460,str,datetime64[us]


In [19]:
print(f"Rows removed overall: {len(df_before) - len(df)} "
      f"({rows_dropped_missing} unrecoverable Total Spent, {duplicates_removed} duplicates)")
print("'Unknown' labels kept as a category:")
print({c: int((df[c] == 'Unknown').sum()) for c in ['Item', 'Payment Method', 'Location']})

Rows removed overall: 23 (23 unrecoverable Total Spent, 0 duplicates)
'Unknown' labels kept as a category:
{'Item': 477, 'Payment Method': 3169, 'Location': 3953}


## 10 · Save the cleaned dataset

In [20]:
df.to_csv(CLEAN_PATH, index=False)

# Round-trip check: reload and confirm shape and a clean parse
chk = pd.read_csv(CLEAN_PATH, parse_dates=["Transaction Date"])
print("Saved:", CLEAN_PATH, chk.shape)
chk.head(10)

Saved:

 clean_cafe_sales.csv (9977, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,Unknown,Unknown,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-Store,2023-06-11
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,Unknown,2023-03-31
6,TXN_4433211,Unknown,3.0,3.0,9.0,Unknown,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,Unknown,2023-10-28
8,TXN_4717867,Unknown,5.0,3.0,15.0,Unknown,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,Unknown,In-Store,2023-12-31


## Summary of decisions

1. **Missing values were hidden** as `''`, `UNKNOWN` and `ERROR`; all were converted to real nulls first.
2. **Exact recovery before imputation:** Price, Quantity, Total and (where unambiguous) Item were reconstructed from the business rules (fixed menu prices and `Total = Qty × Price`), so no statistical guessing was needed for them.
3. **Payment Method / Location** were labelled `Unknown` rather than mode-imputed, to avoid fabricating transactions.
4. **Dates** with no value were left as `NaT` (no valid fill strategy for unordered data).
5. **Rows with no recoverable revenue** were deleted.
6. **Duplicates:** none in the raw file; the check is built in.
7. **Outliers** were flagged by IQR and Z-score but retained — they are valid menu combinations.
8. **Types** were corrected to string / category / Int64 / float / datetime.